In [1]:
#cell1
# Mount Google Drive and define all project paths.

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive")

PROJECT_DIR = DRIVE_ROOT / "final_project"
SOURCE_RAG_DIR = PROJECT_DIR / "RAG"
WORK_DIR = PROJECT_DIR / "logicRAG"

DATA_DIR = WORK_DIR / "data"
EMBEDDINGS_DIR = WORK_DIR / "embeddings"
CACHE_DIR = WORK_DIR / "cache"
REPO_DIR = WORK_DIR / "repo"

for path in [WORK_DIR, DATA_DIR, EMBEDDINGS_DIR, CACHE_DIR, REPO_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("SOURCE_RAG_DIR:", SOURCE_RAG_DIR)
print("WORK_DIR:", WORK_DIR)
print("DATA_DIR:", DATA_DIR)
print("EMBEDDINGS_DIR:", EMBEDDINGS_DIR)
print("CACHE_DIR:", CACHE_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PROJECT_DIR: /content/drive/MyDrive/final_project
SOURCE_RAG_DIR: /content/drive/MyDrive/final_project/RAG
WORK_DIR: /content/drive/MyDrive/final_project/logicRAG
DATA_DIR: /content/drive/MyDrive/final_project/logicRAG/data
EMBEDDINGS_DIR: /content/drive/MyDrive/final_project/logicRAG/embeddings
CACHE_DIR: /content/drive/MyDrive/final_project/logicRAG/cache


In [2]:
#cell2
# Clone the official LogicRAG repository.
# This notebook uses the repository's own BaseRAG class to compute embeddings.

import subprocess
import os
from pathlib import Path

REPO_URL = "https://github.com/chensyCN/LogicRAG.git"
REPO_PATH = REPO_DIR / "LogicRAG"

if not REPO_PATH.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_PATH)], check=True)
else:
    print("Repository already exists. Pulling latest changes...")
    subprocess.run(["git", "-C", str(REPO_PATH), "pull"], check=True)

print("REPO_PATH:", REPO_PATH)

Repository already exists. Pulling latest changes...
REPO_PATH: /content/drive/MyDrive/final_project/logicRAG/repo/LogicRAG


In [3]:
#cell3
# Install only the packages needed for the embedding step.
# No OpenAI API key is needed for this notebook.

!pip -q install -U sentence-transformers tqdm

In [4]:
#cell4
# Copy the original chunk JSON files into final_project/logicRAG/data.
# The original files are kept unchanged.

import shutil

DATASETS = {
    "2wiki": {
        "dataset": "2wikimultihopqa",
        "source_json": SOURCE_RAG_DIR / "2wikimultihopqa_docs_chunks.json",
        "raw_copy_json": DATA_DIR / "2wikimultihopqa_docs_chunks.json",
        "repo_corpus_json": DATA_DIR / "2wikimultihopqa_logicrag_corpus.json",
        "embedding_prefix": "2wiki",
        "cache_dir": CACHE_DIR / "2wiki",
        "manifest_json": EMBEDDINGS_DIR / "2wiki_manifest.json",
    },
    "hotpot": {
        "dataset": "hotpotqa",
        "source_json": SOURCE_RAG_DIR / "hotpotqa_docs_chunks.json",
        "raw_copy_json": DATA_DIR / "hotpotqa_docs_chunks.json",
        "repo_corpus_json": DATA_DIR / "hotpotqa_logicrag_corpus.json",
        "embedding_prefix": "hotpot",
        "cache_dir": CACHE_DIR / "hotpot",
        "manifest_json": EMBEDDINGS_DIR / "hotpot_manifest.json",
    },
}

for name, cfg in DATASETS.items():
    if not cfg["source_json"].exists():
        raise FileNotFoundError(f"Missing source file for {name}: {cfg['source_json']}")

    shutil.copy2(cfg["source_json"], cfg["raw_copy_json"])
    cfg["cache_dir"].mkdir(parents=True, exist_ok=True)

    print(f"{name} copied:")
    print("  source:", cfg["source_json"])
    print("  copy:  ", cfg["raw_copy_json"])

2wiki copied:
  source: /content/drive/MyDrive/final_project/RAG/2wikimultihopqa_docs_chunks.json
  copy:   /content/drive/MyDrive/final_project/logicRAG/data/2wikimultihopqa_docs_chunks.json
hotpot copied:
  source: /content/drive/MyDrive/final_project/RAG/hotpotqa_docs_chunks.json
  copy:   /content/drive/MyDrive/final_project/logicRAG/data/hotpotqa_docs_chunks.json


In [5]:
#cell5
# Convert your chunk schema to the exact corpus schema expected by the repository.
# The repository expects each document to have lowercase keys: title and text.
# Extra metadata is preserved but does not affect embeddings.

import json
from typing import Dict, Any, List

def get_value(row: Dict[str, Any], *keys, default=""):
    for key in keys:
        if key in row and row[key] is not None:
            return row[key]
    return default

def convert_chunks_to_logicrag_corpus(input_json: Path, output_json: Path) -> List[Dict[str, Any]]:
    with open(input_json, "r", encoding="utf-8") as f:
        rows = json.load(f)

    corpus = []
    for i, row in enumerate(rows):
        title = str(get_value(row, "title", "Title", default="")).strip()
        text = str(get_value(row, "text", "Text", "content", default="")).strip()
        chunk_id = str(get_value(row, "chunk_id", "Chunk_id", "id", default=str(i))).strip()

        corpus.append({
            "title": title,
            "text": text,
            "chunk_id": chunk_id,
            "source_index": i,
            "paragraph_id": get_value(row, "paragraph_id", "Paragraph_id", default=None),
            "token_count": get_value(row, "token_count", "Token_count", default=None),
        })

    with open(output_json, "w", encoding="utf-8") as f:
        json.dump(corpus, f, ensure_ascii=False, indent=2)

    return corpus

for name, cfg in DATASETS.items():
    corpus = convert_chunks_to_logicrag_corpus(cfg["raw_copy_json"], cfg["repo_corpus_json"])
    print("=" * 100)
    print("Dataset:", name)
    print("Number of chunks:", len(corpus))
    print("Repository-compatible corpus:", cfg["repo_corpus_json"])
    print("First item:")
    print(json.dumps(corpus[0], ensure_ascii=False, indent=2)[:1500])

Dataset: 2wiki
Number of chunks: 12685
Repository-compatible corpus: /content/drive/MyDrive/final_project/logicRAG/data/2wikimultihopqa_logicrag_corpus.json
First item:
{
  "title": "Calloway County High School",
  "text": "Calloway County High School is a public high school located in Murray, Kentucky. The school was formed from the consolidation of six high schools from across the county: Hazel High School, Lynn Grove High School, Kirksey High School, Almo High School, New Concord High School, and Faxon High School.\nOrganizations: Clubs/Organizations\nState champions: Wrestling: David Woods 195 lbs (2017) Bass Fishing: Bracken Robertson & Dillon Starks (2013) Boys Cross Country: 1984 (2A) Fast Pitch Softball: 2004 Girls Golf: 2012 (Individual, Anna Hack)\nW. Earl Brown, actor: Pookie Jones, 1989 KHSAA Mr. Football winner*",
  "chunk_id": "2wikimultihopqa_chunk_00000001",
  "source_index": 0,
  "paragraph_id": [
    1,
    2,
    3,
    4
  ],
  "token_count": 153
}
Dataset: hotpot
N

In [6]:
#cell6
# Define helper functions for hashing the exact embedded strings and writing manifests.
# The hash is computed from the exact strings used by the repository:
# Title: {title}. Content: {text}

import hashlib
import json
from pathlib import Path

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
MODEL_SHORT_NAME = EMBEDDING_MODEL.split("/")[-1]
NORMALIZED = False
HASH_CHARS_IN_FILENAME = 12

def build_repository_embedding_texts(corpus_json: Path) -> List[str]:
    with open(corpus_json, "r", encoding="utf-8") as f:
        docs = json.load(f)

    return [
        f"Title: {doc['title']}. Content: {doc['text']}"
        for doc in docs
    ]

def compute_exact_corpus_hash(texts: List[str]) -> str:
    h = hashlib.sha256()
    for text in texts:
        h.update(text.encode("utf-8"))
        h.update(b"\n")
    return h.hexdigest()

def write_manifest(
    manifest_path: Path,
    dataset: str,
    num_chunks: int,
    embedding_dim: int,
    corpus_hash: str,
    repo_cache_file: Path,
    named_embedding_file: Path,
):
    manifest = {
        "dataset": dataset,
        "embedding_model": EMBEDDING_MODEL,
        "num_chunks": int(num_chunks),
        "embedding_dim": int(embedding_dim),
        "corpus_hash": corpus_hash,
        "normalized": NORMALIZED,
        "repository_cache_file": str(repo_cache_file),
        "named_embedding_file": str(named_embedding_file),
        "embedding_text_format": "Title: {title}. Content: {text}",
        "note": "Embeddings are generated by the repository BaseRAG code without normalize_embeddings=True.",
    }

    with open(manifest_path, "w", encoding="utf-8") as f:
        json.dump(manifest, f, ensure_ascii=False, indent=2)

    return manifest

In [7]:
#cell7
# Load the same embedding model used in the LogicRAG repository.
# This cell does not import LogicRAG, BaseRAG, OpenAI, or any generation code.

import torch
from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
EMBEDDING_BATCH_SIZE = 32

device = "cuda" if torch.cuda.is_available() else "cpu"

embedder = SentenceTransformer(EMBEDDING_MODEL, device=device)

print("Device:", device)
print("Embedding model:", EMBEDDING_MODEL)
print("Embedding batch size:", EMBEDDING_BATCH_SIZE)
print("Model max sequence length:", embedder.max_seq_length)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Device: cuda
Embedding model: sentence-transformers/all-MiniLM-L6-v2
Embedding batch size: 32
Model max sequence length: 256


In [8]:
#cell8
# Generate embeddings with the same logic as the repository's BaseRAG.encode_sentences_batch.
# Important:
# - No OpenAI API key is required.
# - No generation model is used.
# - No normalize_embeddings=True is used, because the repository does not normalize here.
# - Retrieval later should use cosine_similarity, exactly like the repository.

import torch
import shutil
import gc
import json
from tqdm.auto import tqdm
from pathlib import Path

FORCE_RECOMPUTE = False

def encode_sentences_like_repository(sentences, batch_size=EMBEDDING_BATCH_SIZE):
    all_embeddings = []

    for i in tqdm(range(0, len(sentences), batch_size), desc="Encoding sentences"):
        batch = sentences[i:i + batch_size]

        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        with torch.no_grad():
            embeddings = embedder.encode(
                batch,
                convert_to_tensor=True,
                show_progress_bar=False
            )
            embeddings = embeddings.cpu()
            all_embeddings.append(embeddings)

    final_embeddings = torch.cat(all_embeddings, dim=0).contiguous()

    del all_embeddings
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return final_embeddings

def generate_embeddings_with_repository_logic(dataset_key: str):
    cfg = DATASETS[dataset_key]

    # These are exactly the strings that the repository would embed.
    texts = build_repository_embedding_texts(cfg["repo_corpus_json"])
    num_chunks = len(texts)
    corpus_hash = compute_exact_corpus_hash(texts)
    hash_prefix = corpus_hash[:HASH_CHARS_IN_FILENAME]

    # Repository-compatible cache name.
    repo_cache_file = cfg["cache_dir"] / f"embeddings_{num_chunks}.pt"

    # Your experiment-tracking file name.
    named_embedding_file = EMBEDDINGS_DIR / f"{cfg['embedding_prefix']}_{MODEL_SHORT_NAME}_{hash_prefix}.pt"

    if FORCE_RECOMPUTE:
        if repo_cache_file.exists():
            repo_cache_file.unlink()
        if named_embedding_file.exists():
            named_embedding_file.unlink()

    print("=" * 100)
    print("Dataset:", cfg["dataset"])
    print("Corpus:", cfg["repo_corpus_json"])
    print("Number of chunks:", num_chunks)
    print("Corpus hash:", corpus_hash)
    print("Repository-compatible cache file:", repo_cache_file)
    print("Named embedding file:", named_embedding_file)

    if repo_cache_file.exists() and not FORCE_RECOMPUTE:
        print("Loading existing repository-compatible embedding cache.")
        embeddings = torch.load(repo_cache_file, map_location="cpu")
    else:
        print("Computing embeddings with repository-compatible logic.")
        embeddings = encode_sentences_like_repository(
            sentences=texts,
            batch_size=EMBEDDING_BATCH_SIZE
        )
        torch.save(embeddings, repo_cache_file)

    # Save a second copy with hash in the filename for easier tracking.
    torch.save(embeddings, named_embedding_file)

    manifest = write_manifest(
        manifest_path=cfg["manifest_json"],
        dataset=cfg["dataset"],
        num_chunks=embeddings.shape[0],
        embedding_dim=embeddings.shape[1],
        corpus_hash=corpus_hash,
        repo_cache_file=repo_cache_file,
        named_embedding_file=named_embedding_file,
    )

    print("Embedding shape:", tuple(embeddings.shape))
    print("Embedding dtype:", embeddings.dtype)
    print("Manifest:")
    print(json.dumps(manifest, ensure_ascii=False, indent=2))

    return {
        "embeddings": embeddings,
        "repo_cache_file": repo_cache_file,
        "named_embedding_file": named_embedding_file,
        "manifest": manifest,
    }

# Keep the old function name so cell9 and cell10 do not need to change.
generate_embeddings_with_repository = generate_embeddings_with_repository_logic

In [9]:
#cell9
# Generate embeddings for 2WikiMultiHopQA only.
# This dataset is processed and cached separately.

result_2wiki = generate_embeddings_with_repository("2wiki")

Dataset: 2wikimultihopqa
Corpus: /content/drive/MyDrive/final_project/logicRAG/data/2wikimultihopqa_logicrag_corpus.json
Number of chunks: 12685
Corpus hash: 2be02d631ee4f0715d96ff0471ccc2c83fa68b9784b92643b30c8f587626e6df
Repository-compatible cache file: /content/drive/MyDrive/final_project/logicRAG/cache/2wiki/embeddings_12685.pt
Named embedding file: /content/drive/MyDrive/final_project/logicRAG/embeddings/2wiki_all-MiniLM-L6-v2_2be02d631ee4.pt
Computing embeddings with repository-compatible logic.


Encoding sentences:   0%|          | 0/397 [00:00<?, ?it/s]

Embedding shape: (12685, 384)
Embedding dtype: torch.float32
Manifest:
{
  "dataset": "2wikimultihopqa",
  "embedding_model": "sentence-transformers/all-MiniLM-L6-v2",
  "num_chunks": 12685,
  "embedding_dim": 384,
  "corpus_hash": "2be02d631ee4f0715d96ff0471ccc2c83fa68b9784b92643b30c8f587626e6df",
  "normalized": false,
  "repository_cache_file": "/content/drive/MyDrive/final_project/logicRAG/cache/2wiki/embeddings_12685.pt",
  "named_embedding_file": "/content/drive/MyDrive/final_project/logicRAG/embeddings/2wiki_all-MiniLM-L6-v2_2be02d631ee4.pt",
  "embedding_text_format": "Title: {title}. Content: {text}",
  "note": "Embeddings are generated by the repository BaseRAG code without normalize_embeddings=True."
}


In [10]:
#cell10
# Generate embeddings for HotpotQA only.
# This dataset is processed and cached separately.

result_hotpot = generate_embeddings_with_repository("hotpot")

Dataset: hotpotqa
Corpus: /content/drive/MyDrive/final_project/logicRAG/data/hotpotqa_logicrag_corpus.json
Number of chunks: 35029
Corpus hash: 72a606751ea9bc34eb9290b005bbfc067cb009ee8e5d865c7ad35820e7e50d33
Repository-compatible cache file: /content/drive/MyDrive/final_project/logicRAG/cache/hotpot/embeddings_35029.pt
Named embedding file: /content/drive/MyDrive/final_project/logicRAG/embeddings/hotpot_all-MiniLM-L6-v2_72a606751ea9.pt
Computing embeddings with repository-compatible logic.


Encoding sentences:   0%|          | 0/1095 [00:00<?, ?it/s]

Embedding shape: (35029, 384)
Embedding dtype: torch.float32
Manifest:
{
  "dataset": "hotpotqa",
  "embedding_model": "sentence-transformers/all-MiniLM-L6-v2",
  "num_chunks": 35029,
  "embedding_dim": 384,
  "corpus_hash": "72a606751ea9bc34eb9290b005bbfc067cb009ee8e5d865c7ad35820e7e50d33",
  "normalized": false,
  "repository_cache_file": "/content/drive/MyDrive/final_project/logicRAG/cache/hotpot/embeddings_35029.pt",
  "named_embedding_file": "/content/drive/MyDrive/final_project/logicRAG/embeddings/hotpot_all-MiniLM-L6-v2_72a606751ea9.pt",
  "embedding_text_format": "Title: {title}. Content: {text}",
  "note": "Embeddings are generated by the repository BaseRAG code without normalize_embeddings=True."
}


In [11]:
#cell11
# Validate saved tensors and manifests.
# Since this follows the repository exactly, embedding norms are not expected to be exactly 1.

import torch
import json

def validate_outputs(dataset_key: str):
    cfg = DATASETS[dataset_key]

    with open(cfg["manifest_json"], "r", encoding="utf-8") as f:
        manifest = json.load(f)

    repo_cache_file = Path(manifest["repository_cache_file"])
    named_embedding_file = Path(manifest["named_embedding_file"])

    if not repo_cache_file.exists():
        raise FileNotFoundError(f"Missing repository cache file: {repo_cache_file}")
    if not named_embedding_file.exists():
        raise FileNotFoundError(f"Missing named embedding file: {named_embedding_file}")

    repo_embeddings = torch.load(repo_cache_file, map_location="cpu")
    named_embeddings = torch.load(named_embedding_file, map_location="cpu")

    assert repo_embeddings.shape == named_embeddings.shape
    assert torch.equal(repo_embeddings, named_embeddings)
    assert repo_embeddings.shape[0] == manifest["num_chunks"]
    assert repo_embeddings.shape[1] == manifest["embedding_dim"]
    assert manifest["embedding_model"] == EMBEDDING_MODEL
    assert manifest["normalized"] is False

    sample = repo_embeddings[: min(1000, repo_embeddings.shape[0])]
    norms = torch.linalg.norm(sample, dim=1)

    print("=" * 100)
    print("Dataset:", manifest["dataset"])
    print("Shape:", tuple(repo_embeddings.shape))
    print("Repository cache:", repo_cache_file)
    print("Named embedding:", named_embedding_file)
    print("Manifest:", cfg["manifest_json"])
    print("Sample norms:")
    print("  min:", float(norms.min()))
    print("  max:", float(norms.max()))
    print("  mean:", float(norms.mean()))
    print("Validation passed.")

validate_outputs("2wiki")
validate_outputs("hotpot")

Dataset: 2wikimultihopqa
Shape: (12685, 384)
Repository cache: /content/drive/MyDrive/final_project/logicRAG/cache/2wiki/embeddings_12685.pt
Named embedding: /content/drive/MyDrive/final_project/logicRAG/embeddings/2wiki_all-MiniLM-L6-v2_2be02d631ee4.pt
Manifest: /content/drive/MyDrive/final_project/logicRAG/embeddings/2wiki_manifest.json
Sample norms:
  min: 0.9999998211860657
  max: 1.0000001192092896
  mean: 1.0
Validation passed.
Dataset: hotpotqa
Shape: (35029, 384)
Repository cache: /content/drive/MyDrive/final_project/logicRAG/cache/hotpot/embeddings_35029.pt
Named embedding: /content/drive/MyDrive/final_project/logicRAG/embeddings/hotpot_all-MiniLM-L6-v2_72a606751ea9.pt
Manifest: /content/drive/MyDrive/final_project/logicRAG/embeddings/hotpot_manifest.json
Sample norms:
  min: 0.9999998807907104
  max: 1.0000001192092896
  mean: 1.0
Validation passed.


In [12]:
#cell12
# Run a retrieval smoke test without importing the repository.
# This uses the same retrieval logic as BaseRAG.retrieve:
# 1) encode query
# 2) compute torch.nn.functional.cosine_similarity
# 3) return top-k corpus strings

import torch.nn.functional as F

def retrieval_smoke_test_without_api(dataset_key: str, query: str, top_k: int = 3):
    cfg = DATASETS[dataset_key]

    texts = build_repository_embedding_texts(cfg["repo_corpus_json"])

    with open(cfg["manifest_json"], "r", encoding="utf-8") as f:
        manifest = json.load(f)

    embedding_file = Path(manifest["repository_cache_file"])
    corpus_embeddings = torch.load(embedding_file, map_location="cpu")

    with torch.no_grad():
        query_embedding = embedder.encode(
            [query],
            convert_to_tensor=True,
            show_progress_bar=False
        )[0].cpu()

    similarities = F.cosine_similarity(
        query_embedding.unsqueeze(0),
        corpus_embeddings
    )

    top_scores, top_indices = similarities.topk(top_k)

    print("=" * 100)
    print("Dataset:", cfg["dataset"])
    print("Query:", query)
    print("Top-k:", top_k)
    print()

    for rank, idx in enumerate(top_indices.tolist(), start=1):
        print(f"Rank {rank} | score={float(top_scores[rank-1]):.4f}")
        print(texts[idx][:1000])
        print("-" * 100)

retrieval_smoke_test_without_api(
    "2wiki",
    query="Who is the mother of Lothair II?",
    top_k=3
)

retrieval_smoke_test_without_api(
    "hotpot",
    query="What is one of the stars of The Newcomers known for?",
    top_k=3
)

Dataset: 2wikimultihopqa
Query: Who is the mother of Lothair II?
Top-k: 3

Rank 1 | score=0.5031
Title: Waldrada of Lotharingia. Content: Waldrada was the mistress, and later the wife, of Lothair II of Lotharingia.
Biography: Waldrada's family origin is uncertain. A prolific 19th-century French writer Baron Ernouf suggested that Waldrada was of noble Gallo-Roman family, Baron Ernouf (1858) Histoire de Waldrade, de Lother II et de leurs descendants, p. 3 sister of Thietgaud, the bishop of Trier, and niece of Ghunter, the archbishop of Cologne. Baron Ernouf (1858) Histoire de Waldrade, de Lother II et de leurs descendants, p. 5 . However, these suggestions are not supported by any evidence, and more recent studies have instead suggested she was of relatively undistinguished social origins, though still from an aristocratic milieu. K. Schmid, Ein karolingischer Königseintrag im Gedenkbuch von Remiremont, Frühmittelalterliche Studien, 2, 1968, pp. 96-134 The Vita Sancti Deicoli states that

In [13]:
#cell13
# Print final output layout.

print("Final files:")
print()

for name, cfg in DATASETS.items():
    print("=" * 100)
    print("Dataset:", cfg["dataset"])
    print("Original copied chunks:", cfg["raw_copy_json"])
    print("Repository-compatible corpus:", cfg["repo_corpus_json"])
    print("Dataset cache dir:", cfg["cache_dir"])
    print("Manifest:", cfg["manifest_json"])

    with open(cfg["manifest_json"], "r", encoding="utf-8") as f:
        manifest = json.load(f)

    print("Repository cache file:", manifest["repository_cache_file"])
    print("Named embedding file:", manifest["named_embedding_file"])

print()
print("All embedding files:")
for p in sorted(EMBEDDINGS_DIR.glob("*.pt")):
    print(p)

print()
print("All manifest files:")
for p in sorted(EMBEDDINGS_DIR.glob("*manifest.json")):
    print(p)

Final files:

Dataset: 2wikimultihopqa
Original copied chunks: /content/drive/MyDrive/final_project/logicRAG/data/2wikimultihopqa_docs_chunks.json
Repository-compatible corpus: /content/drive/MyDrive/final_project/logicRAG/data/2wikimultihopqa_logicrag_corpus.json
Dataset cache dir: /content/drive/MyDrive/final_project/logicRAG/cache/2wiki
Manifest: /content/drive/MyDrive/final_project/logicRAG/embeddings/2wiki_manifest.json
Repository cache file: /content/drive/MyDrive/final_project/logicRAG/cache/2wiki/embeddings_12685.pt
Named embedding file: /content/drive/MyDrive/final_project/logicRAG/embeddings/2wiki_all-MiniLM-L6-v2_2be02d631ee4.pt
Dataset: hotpotqa
Original copied chunks: /content/drive/MyDrive/final_project/logicRAG/data/hotpotqa_docs_chunks.json
Repository-compatible corpus: /content/drive/MyDrive/final_project/logicRAG/data/hotpotqa_logicrag_corpus.json
Dataset cache dir: /content/drive/MyDrive/final_project/logicRAG/cache/hotpot
Manifest: /content/drive/MyDrive/final_proje